<div dir="rtl">

# الدرس 48: كتلة الترانسفورمر، ومعمارية جي بي تي

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

### الأجزاء

</div>

In [ ]:
import math
from dataclasses import dataclass
import torch
import torch.nn.functional as F
from torch import nn

@dataclass
class GPTConfig:
    vocab_size: int = 50257
    block_size: int = 1024          # context window
    n_layer: int = 12
    n_head: int = 12
    d_model: int = 768
    dropout: float = 0.0

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg.n_head
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.dropout = cfg.dropout

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=self.dropout if self.training else 0.0)
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc = nn.Linear(cfg.d_model, 4 * cfg.d_model)
        self.proj = nn.Linear(4 * cfg.d_model, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        return self.drop(self.proj(F.gelu(self.fc(x))))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.mlp = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))      # tokens communicate
        x = x + self.mlp(self.ln2(x))       # each token computes
        return x

<div dir="rtl">

### النموذج

</div>

In [ ]:
class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight            # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, m):                            # GPT-2 style initialization
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.block_size, "sequence longer than the context window"
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for block in self.blocks:
            x = block(x)
        logits = self.head(self.ln_f(x))                   # (B, T, vocab_size)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

<div dir="rtl">

### عدّ معاملات جي بي تي 2، بدون بنائه في الذاكرة

</div>

In [ ]:
with torch.device("meta"):
    gpt2 = GPT(GPTConfig())
total = sum(p.numel() for p in gpt2.parameters())      # the tied matrix is counted once
per_block = sum(p.numel() for p in gpt2.blocks[0].parameters())
print(f"GPT-2 small: {total:,} parameters | one block: {per_block:,}")

<div dir="rtl">

### نموذج صغير، وأول تمرير

</div>

In [ ]:
torch.manual_seed(0)
cfg = GPTConfig(vocab_size=65, block_size=64, n_layer=4, n_head=4, d_model=128)
model = GPT(cfg)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

idx = torch.randint(0, 65, (2, 64))                  # 2 random sequences of 64 tokens
targets = torch.randint(0, 65, (2, 64))              # random "next tokens"
logits, loss = model(idx, targets)
print("logits shape:", tuple(logits.shape))
print(f"initial loss {loss.item():.3f}   vs   ln(65) = {math.log(65):.3f}")

<div dir="rtl">

### دالة التفعيل

</div>

In [ ]:
import matplotlib.pyplot as plt
z = torch.linspace(-4, 4, 400)
plt.figure(figsize=(6, 3.2))
plt.plot(z, F.relu(z), label="ReLU", lw=2)
plt.plot(z, F.gelu(z), label="GELU", lw=2)
plt.axhline(0, color="gray", lw=0.8); plt.axvline(0, color="gray", lw=0.8)
plt.legend(); plt.grid(True, alpha=0.4)
plt.show()

In [ ]:
print(F.gelu(torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])))

<div dir="rtl">

## تمرين: معاملات نموذجك

نموذج الحروف في الكود: 65 رمزاً، وسياق 64، و 4 كتل، و 4 رؤوس، وطول 128.

1. احسب بيدك معاملات: تضمين الرموز، وتضمين المواضع، وكتلة واحدة، والتطبيع الأخير.
2. اجمعها، وتذكّر أن الرأس مربوط.
3. تحقق من رقم الكود.

</div>

In [ ]:
# اكتب حلّك هنا
